In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from catboost import CatBoostClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier    ### Combine classifiers predictions
)
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1) DATA EXPLORATION

In [2]:
df = pd.read_parquet("train.parquet")
print(df.shape)
df.info()

(17499636, 19)
<class 'pandas.core.frame.DataFrame'>
Index: 17499636 entries, 0 to 25661583
Data columns (total 19 columns):
 #   Column         Dtype         
---  ------         -----         
 0   status         int64         
 1   gender         object        
 2   firstName      object        
 3   level          object        
 4   lastName       object        
 5   userId         object        
 6   ts             int64         
 7   auth           object        
 8   page           object        
 9   sessionId      int64         
 10  location       object        
 11  itemInSession  int64         
 12  userAgent      object        
 13  method         object        
 14  length         float64       
 15  song           object        
 16  artist         object        
 17  time           datetime64[us]
 18  registration   datetime64[us]
dtypes: datetime64[us](2), float64(1), int64(4), object(12)
memory usage: 2.6+ GB


In [3]:
print(df.status.value_counts())
df.head(30)


status
200    16020693
307     1461649
404       17294
Name: count, dtype: int64


,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
0,200,M,Shlok,paid,Johnson,1749042,1538352001000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",278,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,524.32934,Ich mache einen Spiegel - Dream Part 4,Popol Vuh,2018-10-01 00:00:01,2018-08-08 13:22:21
992,200,M,Shlok,paid,Johnson,1749042,1538352525000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",279,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,178.02404,Monster (Album Version),Skillet,2018-10-01 00:08:45,2018-08-08 13:22:21
1360,200,M,Shlok,paid,Johnson,1749042,1538352703000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",280,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,232.61995,Seven Nation Army,The White Stripes,2018-10-01 00:11:43,2018-08-08 13:22:21
1825,200,M,Shlok,paid,Johnson,1749042,1538352935000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",281,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,265.50812,Under The Bridge (Album Version),Red Hot Chili Peppers,2018-10-01 00:15:35,2018-08-08 13:22:21
2366,200,M,Shlok,paid,Johnson,1749042,1538353200000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",282,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,471.69261,Circlesong 6,Bobby McFerrin,2018-10-01 00:20:00,2018-08-08 13:22:21
3271,200,M,Shlok,paid,Johnson,1749042,1538353671000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",283,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,266.86649,Who Can Compare,Foolish Things,2018-10-01 00:27:51,2018-08-08 13:22:21
3802,200,M,Shlok,paid,Johnson,1749042,1538353937000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",284,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,1400.65914,Angel Dust,Gil Scott Heron,2018-10-01 00:32:17,2018-08-08 13:22:21
6585,200,M,Shlok,paid,Johnson,1749042,1538355337000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",285,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,186.98404,Sweet And Dandy,Toots & The Maytals,2018-10-01 00:55:37,2018-08-08 13:22:21
6675,200,M,Shlok,paid,Johnson,1749042,1538355388000,Logged In,Downgrade,22683,"Dallas-Fort Worth-Arlington, TX",286,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",GET,NaN,None,None,2018-10-01 00:56:28,2018-08-08 13:22:21
6961,200,M,Shlok,paid,Johnson,1749042,1538355523000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",287,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,306.05016,On The Moon,Peter Cincotti,2018-10-01 00:58:43,2018-08-08 13:22:21


In [4]:
X= df[df['userId'] == '1749042']
X[X['status'] != 200]

,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
24330,307,M,Shlok,paid,Johnson,1749042,1538363406000,Logged In,Thumbs Up,22683,"Dallas-Fort Worth-Arlington, TX",324,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-01 03:10:06,2018-08-08 13:22:21
391536,307,M,Shlok,paid,Johnson,1749042,1538467226000,Logged In,Add Friend,25570,"Dallas-Fort Worth-Arlington, TX",4,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-02 08:00:26,2018-08-08 13:22:21
883667,307,M,Shlok,paid,Johnson,1749042,1538591742000,Logged In,Thumbs Down,31304,"Dallas-Fort Worth-Arlington, TX",6,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 18:35:42,2018-08-08 13:22:21
893741,307,M,Shlok,paid,Johnson,1749042,1538593847000,Logged In,Thumbs Down,31304,"Dallas-Fort Worth-Arlington, TX",18,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 19:10:47,2018-08-08 13:22:21
916990,307,M,Shlok,paid,Johnson,1749042,1538598939000,Logged In,Thumbs Up,31304,"Dallas-Fort Worth-Arlington, TX",37,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 20:35:39,2018-08-08 13:22:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6641195,307,M,Shlok,paid,Johnson,1749042,1540076041000,Logged In,Thumbs Up,100362,"Dallas-Fort Worth-Arlington, TX",16,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-20 22:54:01,2018-08-08 13:22:21
6651435,307,M,Shlok,paid,Johnson,1749042,1540080154000,Logged In,Save Settings,100362,"Dallas-Fort Worth-Arlington, TX",37,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 00:02:34,2018-08-08 13:22:21
6658537,307,M,Shlok,paid,Johnson,1749042,1540082944000,Logged In,Thumbs Up,100362,"Dallas-Fort Worth-Arlington, TX",51,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 00:49:04,2018-08-08 13:22:21
6660411,307,M,Shlok,paid,Johnson,1749042,1540083704000,Logged In,Logout,100362,"Dallas-Fort Worth-Arlington, TX",56,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 01:01:44,2018-08-08 13:22:21


In [5]:
COLS = [
    "userId", "time", "registration",
    "page", "sessionId", "level",
    "gender", "length", "artist",
    "song", "status", "method",
    "userAgent", "auth", "firstName", 
    "lastName", "itemInSession", "location"
]

train = pd.read_parquet("train.parquet", columns=COLS)
test = pd.read_parquet("test.parquet", columns=COLS)

for data in (train, test):
    data["userId"] = data["userId"].astype(str)
    data["time"] = pd.to_datetime(data["time"])
    data["registration"] = pd.to_datetime(data["registration"])

# 7. Vérifications
print("Train :", train.shape, "dates :", train.time.min(), train.time.max())
print("Test  :", test.shape, "dates :", test.time.min(), test.time.max())
print("Common Users :", len(set(train.userId) & set(test.userId)))
print(
    "Churn present in test:",
    test.loc[
        test.page.eq("Cancellation Confirmation"), "userId"
    ].nunique())

print("Auth states :")
print(train["auth"].value_counts(dropna=False)) ### No need corresp to page which has more info

print("\nNumber of different User Agents  :")
print(train["userAgent"].nunique())

print("\nUser Agents most frequents :")
print(train["userAgent"].value_counts().head(10))

print("\nUsers with >1 User Agent :")
print(
    (train.groupby("userId")["userAgent"].nunique() > 1).sum()
)

Train : (17499636, 18) dates : 2018-10-01 00:00:01 2018-11-20 00:00:00
Test  : (4393179, 18) dates : 2018-10-01 00:00:06 2018-11-20 00:00:00
Common Users : 0
Churn present in test: 0
Auth states :
auth
Logged In    17495365
Cancelled        4271
Name: count, dtype: int64

Number of different User Agents  :
85

User Agents most frequents :
userAgent
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"               1605877
Mozilla/5.0 (Windows NT 6.1; WOW64; rv:31.0) Gecko/20100101 Firefox/31.0                                                      1365774
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"               1063498
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"    1051132
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537

In [6]:
# Nombre de valeurs distinctes par utilisateur
users_info = train.groupby("userId").agg(
    n_firstnames=("firstName", "nunique"),
    n_lastnames=("lastName", "nunique"),
    n_agents=("userAgent", "nunique"),
    n_locations=("location", "nunique")
)

print("UserId with >1 first name :",
      (users_info["n_firstnames"] > 1).sum())

print("UserId with >1 last name :",
      (users_info["n_lastnames"] > 1).sum())

print("UserId with >1 location :",
      (users_info["n_locations"] > 1).sum())

# Combien de comptes distincts utilisent chaque User Agent ?
print("\nNumber of user by User Agent :")
print(
    train.groupby("userAgent")["userId"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

# Explorer les localisations
print("\nLocalitions most frequent:")
print(train["location"].value_counts().head(10))

# Explorer itemInSession
print("\nitemInSession Distribution:")
print(train["itemInSession"].describe(
    percentiles=[0.5, 0.9, 0.99]
))


UserId with >1 first name : 0
UserId with >1 last name : 0
UserId with >1 location : 0

Number of user by User Agent :
userAgent
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"               1712
Mozilla/5.0 (Windows NT 6.1; WOW64; rv:31.0) Gecko/20100101 Firefox/31.0                                                      1437
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"               1205
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"    1139
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537.78.2"       1046
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"     930
Mozilla/5.0 (Macintosh; Intel Mac OS X 10.9; rv:31.0) Gecko/20100101 Firefox/31.0    

In [7]:
print("Repartition of HTTP statuts:")
print(train["status"].value_counts(dropna=False))

print("\nRepartition of HTTP methods :")
print(train["method"].value_counts(dropna=False))

print("\nHTTP Statuts by page type:")
print(pd.crosstab(train["page"], train["status"]))

Repartition of HTTP statuts:
status
200    16020693
307     1461649
404       17294
Name: count, dtype: int64

Repartition of HTTP methods :
method
PUT    16162688
GET     1336948
Name: count, dtype: int64

HTTP Statuts by page type:
status                          200     307    404
page                                              
About                         33117       0      0
Add Friend                        0  262147      0
Add to Playlist              409606       0      0
Cancel                            0    4271      0
Cancellation Confirmation      4271       0      0
Downgrade                    124248       0      0
Error                             0       0  17294
Help                          89035       0      0
Home                         645259       0      0
Logout                            0  204700      0
NextSong                   14291433       0      0
Roll Advert                  284837       0      0
Save Settings                     0   20370      0
S

In [8]:
print("\nHTTP Methods by page type :")
print(pd.crosstab(train["page"], train["method"]))


HTTP Methods by page type :
method                        GET       PUT
page                                       
About                       33117         0
Add Friend                      0    262147
Add to Playlist                 0    409606
Cancel                          0      4271
Cancellation Confirmation    4271         0
Downgrade                  124248         0
Error                       17294         0
Help                        89035         0
Home                       645259         0
Logout                          0    204700
NextSong                        0  14291433
Roll Advert                284837         0
Save Settings                   0     20370
Settings                   101191         0
Submit Downgrade                0      4425
Submit Upgrade                  0     11381
Thumbs Down                     0    164964
Thumbs Up                       0    789391
Upgrade                     37696         0


In [9]:
# number of events by session
sessions = (
    train.groupby(["userId", "sessionId"])
    .size()
    .reset_index(name="nb_events")
)

# Distribution of the size of the sessions
print(sessions["nb_events"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
))

count    208358.000000
mean         83.988309
std         101.473892
min           1.000000
25%          19.000000
50%          49.000000
75%         109.000000
90%         207.000000
99%         484.000000
max        1393.000000
Name: nb_events, dtype: float64


In [10]:
sessions_dates = train.groupby(
    ["userId", "sessionId"]
)["time"].agg(["min", "max"])

# Sessions qui commencent et finissent à des dates différentes
sessions_multi_days = sessions_dates[
    sessions_dates["min"].dt.date
    != sessions_dates["max"].dt.date
]

print("Nombre total de sessions :", len(sessions_dates))
print("Sessions traversant minuit :", len(sessions_multi_days))

print("\nExemples :")
print(sessions_multi_days.head())

Nombre total de sessions : 208358
Sessions traversant minuit : 39416

Exemples :
                                  min                 max
userId  sessionId                                        
1000025 31688     2018-10-02 18:12:22 2018-10-03 21:31:44
        60612     2018-10-09 23:27:02 2018-10-10 09:26:10
        69552     2018-10-11 18:48:52 2018-10-12 00:21:00
        85743     2018-10-15 19:58:41 2018-10-16 01:40:29
        86823     2018-10-16 22:36:23 2018-10-17 07:39:06


In [11]:
# Durée des sessions en minutes
sessions_dates["duration_min"] = (
    sessions_dates["max"] - sessions_dates["min"]
).dt.total_seconds() / 60

print(sessions_dates["duration_min"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
))

count    208358.000000
mean        288.780987
std         364.305263
min           0.000000
25%          57.416667
50%         159.800000
75%         375.929167
90%         726.916667
99%        1734.817000
max        5790.583333
Name: duration_min, dtype: float64


In [12]:
# Identifier la session la plus longue
user, session = sessions_dates["duration_min"].idxmax()

# Récupérer ses événements
long_session = train[
    (train["userId"] == user) &
    (train["sessionId"] == session)
].sort_values("time").copy()

# Temps écoulé entre deux événements consécutifs
long_session["gap_min"] = (
    long_session["time"].diff().dt.total_seconds() / 60
)

print("Utilisateur :", user)
print("Session :", session)
print("Nombre d'événements :", len(long_session))

print("\nLes 10 plus longues périodes d'inactivité :")
print(
    long_session.nlargest(10, "gap_min")[
        ["time", "page", "itemInSession", "gap_min"]
    ]
)

Utilisateur : 1642682
Session : 89129
Nombre d'événements : 1329

Les 10 plus longues périodes d'inactivité :
                       time             page  itemInSession    gap_min
5766698 2018-10-18 11:29:41             Home            505  48.483333
6611006 2018-10-20 19:45:36        Downgrade           1308  46.466667
6268953 2018-10-19 17:28:12         NextSong            929  36.300000
5293683 2018-10-17 05:31:35             Home             97  35.766667
5319605 2018-10-17 07:27:03  Add to Playlist            121  31.700000
5259286 2018-10-17 03:00:55             Home             58  29.416667
6389370 2018-10-19 23:53:18             Home           1034  29.400000
6543120 2018-10-20 13:09:26  Add to Playlist           1229  29.083333
5326155 2018-10-17 07:55:45         NextSong            122  28.700000
6132828 2018-10-19 09:31:56             Home            818  28.650000


In [13]:

# Dernier événement de chaque session
last_events = train.groupby(
    ["userId", "sessionId"]
)["itemInSession"].max()

# Dernier Logout de chaque session concernée
logout_events = train[
    train["page"] == "Logout"
].groupby(
    ["userId", "sessionId"]
)["itemInSession"].max()

# Sessions avec au moins un Logout
n_logout = len(logout_events)

# Sessions où un événement survient après le dernier Logout
n_after_logout = (
    logout_events < last_events.loc[logout_events.index]
).sum()

print("Nombre total de sessions :", len(last_events))
print("Sessions avec Logout :", n_logout)
print("Sessions sans Logout :", len(last_events) - n_logout)
print("Sessions avec des événements après Logout :", n_after_logout)


Nombre total de sessions : 208358
Sessions avec Logout : 111545
Sessions sans Logout : 96813
Sessions avec des événements après Logout : 78236


In [14]:

# Sessions où des événements suivent le dernier Logout
mask = logout_events < last_events.loc[logout_events.index]

# Sélectionner un exemple
user, session = mask[mask].index[0]

# Récupérer les événements de cette session
example = train[
    (train["userId"] == user) &
    (train["sessionId"] == session)
].sort_values("itemInSession").reset_index(drop=True)

# Position du dernier Logout
logout_pos = example.index[
    example["page"] == "Logout"
][-1]

# Observer les événements avant et après
print(
    example.iloc[
        max(0, logout_pos - 3):logout_pos + 6
    ][["time", "page", "itemInSession"]]
)


                   time      page  itemInSession
408 2018-10-03 17:44:21  NextSong            413
409 2018-10-03 17:49:24  NextSong            414
410 2018-10-03 17:53:38  NextSong            415
411 2018-10-03 17:53:39    Logout            416
412 2018-10-03 17:54:23      Home            420
413 2018-10-03 17:56:16  NextSong            421
414 2018-10-03 18:00:25  NextSong            422
415 2018-10-03 18:03:41  NextSong            423
416 2018-10-03 18:07:00  NextSong            424


In [15]:
# Date de résiliation de chaque utilisateur
cancellations = (
    train[train["page"] == "Cancellation Confirmation"]
    .groupby("userId")["time"]
    .min()
)

# Événements des utilisateurs ayant résilié
churn_events = train[
    train["userId"].isin(cancellations.index)
].copy()

# Associer la date de résiliation à chaque événement
churn_events["cancel_date"] = (
    churn_events["userId"].map(cancellations)
)

# Conserver les événements après résiliation
after_cancel = churn_events[
    churn_events["time"] > churn_events["cancel_date"]
]

print("Utilisateurs actifs après résiliation :",
      after_cancel["userId"].nunique())

print("\nÉvénements après résiliation :")
print(after_cancel["page"].value_counts())


Utilisateurs actifs après résiliation : 0

Événements après résiliation :
Series([], Name: count, dtype: int64)


# 2) Création variable cible

In [16]:
# Horizon de prédiction : 10 jours
HORIZON = pd.Timedelta(days=10)

# Dates auxquelles nous simulons une prédiction
CUTOFFS = pd.date_range(
    start="2018-10-11",
    end="2018-11-10",
    freq="10D"
)

# Affichage de nos fenêtres temporelles
for cutoff in CUTOFFS:
    target_end = cutoff + HORIZON

    print(f"Date de prédiction : {cutoff.date()}")
    print(f"Churn observé entre : {cutoff.date()} et {target_end.date()} (exclu)")
    print("-" * 50)

Date de prédiction : 2018-10-11
Churn observé entre : 2018-10-11 et 2018-10-21 (exclu)
--------------------------------------------------
Date de prédiction : 2018-10-21
Churn observé entre : 2018-10-21 et 2018-10-31 (exclu)
--------------------------------------------------
Date de prédiction : 2018-10-31
Churn observé entre : 2018-10-31 et 2018-11-10 (exclu)
--------------------------------------------------
Date de prédiction : 2018-11-10
Churn observé entre : 2018-11-10 et 2018-11-20 (exclu)
--------------------------------------------------


In [17]:
# Première date de prédiction
cutoff = CUTOFFS[0]
target_end = cutoff + HORIZON

# Date de résiliation de chaque utilisateur
cancellations = (
    train[train["page"] == "Cancellation Confirmation"]
    .groupby("userId")["time"]
    .min()
)

# Utilisateurs connus avant la date de prédiction
users = pd.Index(
    train.loc[train["time"] < cutoff, "userId"].unique()
)

# Exclure les utilisateurs ayant déjà résilié
already_churned = cancellations[
    cancellations < cutoff
].index

users = users[~users.isin(already_churned)]

# Résiliations dans les 10 jours suivants
future_churn = cancellations[
    (cancellations >= cutoff) &
    (cancellations < target_end)
].index

# Création de la variable cible
labels = pd.DataFrame({"userId": users})

labels["churn"] = (
    labels["userId"].isin(future_churn).astype(int)
)

# Vérifications
print("Nombre d'utilisateurs :", len(labels))

print("\nRépartition du churn :")
print(labels["churn"].value_counts())

print("\nTaux de churn :")
print(round(labels["churn"].mean() * 100, 2), "%")

display(labels.head())


Nombre d'utilisateurs : 14176

Répartition du churn :
churn
0    13355
1      821
Name: count, dtype: int64

Taux de churn :
5.79 %


,userId,churn
0,1749042,0
1,1563081,0
2,1697168,0
3,1222580,0
4,1714398,0


In [22]:
# Liste qui contiendra les labels de chaque période
all_labels = []

for cutoff in CUTOFFS:

    target_end = cutoff + HORIZON

    # Utilisateurs connus avant la date de prédiction
    users = pd.Index(
        train.loc[train["time"] < cutoff, "userId"].unique()
    )

    # Exclure ceux qui ont déjà résilié
    already_churned = cancellations[
        cancellations < cutoff
    ].index

    users = users[~users.isin(already_churned)]

    # Utilisateurs qui résilient dans les 10 jours suivants
    future_churn = cancellations[
        (cancellations >= cutoff) &
        (cancellations < target_end)
    ].index

    # Créer les labels de la période
    labels_period = pd.DataFrame({"userId": users})

    labels_period["cutoff"] = cutoff

    labels_period["churn"] = (
        labels_period["userId"]
        .isin(future_churn)
        .astype(int)
    )

    # Ajouter cette période à notre liste
    all_labels.append(labels_period)

# Fusionner toutes les périodes
labels_all = pd.concat(all_labels, ignore_index=True)

# Résumé par période
summary = labels_all.groupby("cutoff").agg(
    n_users=("churn", "size"),
    n_churn=("churn", "sum"),
    churn_rate=("churn", "mean")
)

summary["churn_rate"] *= 100

display(summary)

all_labels


,n_users,n_churn,churn_rate
cutoff,,,
2018-10-11,14176,821,5.791479
2018-10-21,15452,817,5.287341
2018-10-31,15530,789,5.080489
2018-11-10,15276,663,4.340141


[        userId     cutoff  churn
 0      1749042 2018-10-11      0
 1      1563081 2018-10-11      0
 2      1697168 2018-10-11      0
 3      1222580 2018-10-11      0
 4      1714398 2018-10-11      0
 ...        ...        ...    ...
 14171  1208557 2018-10-11      0
 14172  1896647 2018-10-11      0
 14173  1494890 2018-10-11      0
 14174  1729584 2018-10-11      0
 14175  1878214 2018-10-11      0
 
 [14176 rows x 3 columns],
         userId     cutoff  churn
 0      1749042 2018-10-21      1
 1      1563081 2018-10-21      0
 2      1697168 2018-10-21      0
 3      1222580 2018-10-21      1
 4      1714398 2018-10-21      0
 ...        ...        ...    ...
 15447  1777544 2018-10-21      0
 15448  1668254 2018-10-21      0
 15449  1527915 2018-10-21      0
 15450  1510873 2018-10-21      0
 15451  1391436 2018-10-21      0
 
 [15452 rows x 3 columns],
         userId     cutoff  churn
 0      1563081 2018-10-31      0
 1      1697168 2018-10-31      0
 2      1714398 2018-10-

# 3) Feature engeneering

In [ ]:
all_datasets = []

data_start = train["time"].min().floor("D")

for cutoff in CUTOFFS:

    history_start = cutoff - HORIZON

    # 1. HISTORIQUE DES 10 DERNIERS JOURS
    history = train.loc[
        (train["time"] >= history_start) &
        (train["time"] < cutoff),
        [
            "userId", "time", "page", "sessionId",
            "length", "artist", "song"
        ]
    ].copy()

    # 2. INDICATEURS PAR ÉVÉNEMENT
    history["day"] = history["time"].dt.floor("D")
    history["is_song"] = history["page"].eq("NextSong")
    history["is_like"] = history["page"].eq("Thumbs Up")
    history["is_dislike"] = history["page"].eq("Thumbs Down")
    history["is_error"] = history["page"].eq("Error")
    history["is_playlist"] = history["page"].eq("Add to Playlist")
    history["is_downgrade"] = history["page"].eq("Submit Downgrade")

    # 3. FEATURES D'ACTIVITÉ, ENGAGEMENT ET MUSIQUE
    features_period = history.groupby("userId").agg(
        n_events_10d=("page", "size"),
        n_sessions_10d=("sessionId", "nunique"),
        n_active_days_10d=("day", "nunique"),
        n_songs_10d=("is_song", "sum"),
        n_likes_10d=("is_like", "sum"),
        n_dislikes_10d=("is_dislike", "sum"),
        n_errors_10d=("is_error", "sum"),
        n_playlist_adds_10d=("is_playlist", "sum"),
        total_song_length_10d=("length", "sum"),
        n_artists_10d=("artist", "nunique"),
        n_unique_songs_10d=("song", "nunique"),

        # Nouvelle feature : downgrade
        downgraded_10d=("is_downgrade", "max")
    ).reset_index()

    # 4. RÉCUPÉRER LES LABELS DE LA PÉRIODE
    labels_period = labels_all.loc[
        labels_all["cutoff"] == cutoff
    ].copy()

    # 5. FUSION LABELS + FEATURES
    dataset_period = labels_period.merge(
        features_period,
        on="userId",
        how="left",
        validate="one_to_one"
    )

   ### without activity on 10d
    count_cols = features_period.columns.drop("userId")

    dataset_period[count_cols] = (
        dataset_period[count_cols].fillna(0)
    )

    # 6. FEATURE ABONNEMENT ACTUEL
    past = train.loc[
        train["time"] < cutoff,
        ["userId", "time", "level"]
    ]

    ## last event for ea user
    last_idx = past.groupby("userId")["time"].idxmax()

    last_levels = (
        past.loc[last_idx]
        .set_index("userId")["level"]
    )

    # Paid = 1, Free = 0
    dataset_period["is_paid"] = (
        dataset_period["userId"]
        .map(last_levels)
        .map({"paid": 1, "free": 0})
    )

    # 7. FEATURE evol
    previous_start = cutoff - 2 * HORIZON
    previous_end = cutoff - HORIZON

    if previous_start >= data_start:

        # listen in 10 previous days
        previous_songs = train.loc[
            (train["time"] >= previous_start) &
            (train["time"] < previous_end) &
            (train["page"] == "NextSong"),
            "userId"
        ].value_counts()

        n_previous = (
            dataset_period["userId"]
            .map(previous_songs)
            .fillna(0)
        )

        n_recent = dataset_period["n_songs_10d"]

        # var betw -1 and +1
        dataset_period["songs_trend_10d"] = (
            (n_recent - n_previous) /
            (n_recent + n_previous + 1)
        )

    else:
        # indisp elems
        dataset_period["songs_trend_10d"] = np.nan

    
    all_datasets.append(dataset_period)


dataset_all = pd.concat(
    all_datasets,
    ignore_index=True
)

### selection 12 eatures

FEATURES_12 = [
    "n_sessions_10d",
    "n_active_days_10d",
    "n_songs_10d",
    "n_likes_10d",
    "n_dislikes_10d",
    "n_errors_10d",
    "n_playlist_adds_10d",
    "total_song_length_10d",
    "n_artists_10d",
    "downgraded_10d",
    "is_paid",
    "songs_trend_10d"
]

dataset_model = dataset_all[
    ["userId", "cutoff", "churn"] + FEATURES_12
].copy()

### verif

print("Dataset complet :", dataset_all.shape)
print("Dataset avec 12 features :", dataset_model.shape)

print("\nObservations par période :")
print(dataset_model.groupby("cutoff").size())

print("\nValeurs manquantes :")
print(dataset_model[FEATURES_12].isna().sum())

display(dataset_model.head())

Dataset complet : (60434, 17)
Dataset avec 12 features : (60434, 15)

Observations par période :
cutoff
2018-10-11    14176
2018-10-21    15452
2018-10-31    15530
2018-11-10    15276
dtype: int64

Valeurs manquantes :
n_sessions_10d               0
n_active_days_10d            0
n_songs_10d                  0
n_likes_10d                  0
n_dislikes_10d               0
n_errors_10d                 0
n_playlist_adds_10d          0
total_song_length_10d        0
n_artists_10d                0
downgraded_10d               0
is_paid                      0
songs_trend_10d          14176
dtype: int64


,userId,cutoff,churn,n_sessions_10d,n_active_days_10d,n_songs_10d,n_likes_10d,n_dislikes_10d,n_errors_10d,n_playlist_adds_10d,total_song_length_10d,n_artists_10d,downgraded_10d,is_paid,songs_trend_10d
0,1749042,2018-10-11,0,6.0,8.0,486.0,21.0,7.0,0.0,13.0,121710.35590,416.0,False,1,NaN
1,1563081,2018-10-11,0,1.0,1.0,25.0,1.0,0.0,0.0,0.0,6304.72010,23.0,False,1,NaN
2,1697168,2018-10-11,0,9.0,9.0,339.0,15.0,5.0,1.0,8.0,84159.43194,305.0,True,0,NaN
3,1222580,2018-10-11,0,11.0,9.0,1275.0,59.0,12.0,2.0,37.0,320254.37710,957.0,True,1,NaN
4,1714398,2018-10-11,0,4.0,4.0,327.0,19.0,4.0,1.0,10.0,84535.59698,293.0,False,1,NaN


# Separation train, validation and test